# Fragility-based disruption
The available disruption scenarios in the package do not consider the specific characteristics of the hazards and the fragility of the infrastructure components. For more accurate analysis, an external hazard- and fragility analysis model can be used to generate direct disruptions in the networks and then use it as input for network level simulations using InfraRisk. This notebook shows how to generate disruptions if the hazard fields (for e.g., in the case of earthquakes, PGA or PGV) and the fragility curves are provided.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from infrarisk.src.physical.integrated_network import IntegratedNetwork
from infrarisk.src.hazards.fragility_based import FragilityBasedDisruption

import pandas as pd
import geopandas as gpd
import os

import warnings
warnings.filterwarnings('ignore')

### Load the micropolis network

In [2]:
shelby_network = IntegratedNetwork(name = 'Shelby County')

In [3]:
MAIN_DIR = Path('../..')
network_dir= 'infrarisk/data/networks/shelby'
water_folder = MAIN_DIR/f'{network_dir}/water'
power_folder = MAIN_DIR/f'{network_dir}/power'
transp_folder = MAIN_DIR/f'{network_dir}/transportation/reduced'

# load all infrastructure networks
shelby_network.load_networks(water_folder, 
                                 power_folder, 
                                 transp_folder, 
                                 power_sim_type = '1ph', 
                                 water_sim_type = "PDA")
shelby_network.generate_integrated_graph(basemap = True)

Water network successfully loaded from ../../infrarisk/data/networks/shelby/water/water.inp. The analysis type is set to PDA.
initial simulation duration: 60s; hydraulic time step: 60s; pattern time step: 3600s

Loading water service area details...


numba cannot be imported and numba functions are disabled.
Probably the execution is slow.
Please install numba to gain a massive speedup.
(or if you prefer slow execution, set the flag numba=False to avoid this warning!)
numba cannot be imported and numba functions are disabled.
Probably the execution is slow.
Please install numba to gain a massive speedup.
(or if you prefer slow execution, set the flag numba=False to avoid this warning!)


Power system successfully loaded from ../../infrarisk/data/networks/shelby/power/power.json. Single phase power flow simulation will be used.

Loading power service area details...
Transportation network successfully loaded from ../../infrarisk/data/networks/shelby/transportation/reduced. Static traffic assignment method will be used to calculate travel times.
Successfully added power network to the integrated graph...
Successfully added water network to the integrated graph...
Successfully added transportation network to the integrated graph...
Integrated graph successfully created.
Generating betweenness centrality...


Loading BokehJS ...

figure(id='p1003', ...)

### Generate a fragility-based disruption

In [4]:
fragility_df = pd.read_csv(MAIN_DIR/f'infrarisk/data/networks/shelby/fragility_curves/hazus_fragility_and_recovery_low.csv')

earthquake_disruption  = FragilityBasedDisruption(name = "Earthquake disruption",
                                                  fragility_df=fragility_df,
                                                  resilience_level="low",
                                                  time_of_occurrence=3600)
earthquake_disruption.set_fail_compon_dict({
            "power": {"L", "TFEG", "TFLO", "MP"},
            "water": {"PMA", "T", "WP"},
            "transport": {"L"},
        })
earthquake_disruption.set_all_fragility_and_recovery_curves()

In [5]:
gmf = pd.read_csv(MAIN_DIR/f'infrarisk/data/networks/shelby/hazards/earthquake/gmfs/2475/203080_7.7_gms.csv')
gmf_gpd = gpd.GeoDataFrame(gmf, geometry=gpd.points_from_xy(gmf['lon_UTM'], gmf['lat_UTM'])).set_crs('epsg:3857')
earthquake_disruption.plot_imt(gmf_gpd, imt_column = 'PGA')

In [6]:
earthquake_disruption.ascertain_damage_probabilities(component = 'P_L65', imt_type = 'PGA', imt_value = 0.5)

[0.017211125580801068,
 0.8199084148320286,
 0.16115692657648953,
 6.0500521415113146e-05]

In [7]:
earthquake_disruption.ascertain_pipe_damage_probabilities(wn = shelby_network.wn, pipe = 'W_PMA45', gmf_gpd=gmf_gpd)

[nan, nan]

In [8]:
G = shelby_network.integrated_graph
earthquake_disruption.set_gmfs(G,gmf_gpd)

In [9]:
earthquake_disruption.set_affected_components(shelby_network, gmf_gpd, 7200, plot_components=True)
fail_probs_df = earthquake_disruption.get_affected_components()

In [10]:
earthquake_disruption.fail_probs_df.head()

,component,disruption_time,state_probs,disruption_state,recovery_time,infra,damage_perc


In [11]:
earthquake_disruption.plot_failure_distributions()

No failure probability data available to plot.


### Write the disruption file to local directory

In [12]:
scenario_location = MAIN_DIR/network_dir/"scenarios/earthquake3"
earthquake_disruption.generate_disruption_file(location = scenario_location)